In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Change this path to your actual CSV file location
df = pd.read_csv(r"C:\Users\Rishv\Desktop\SNL Lab\Participant Data\CMT0015_MASTER_Summary.csv")
df = df[df['BlockType'] == 'S'].copy()

# Split by direction
df_right = df[(df['Direction'] == 'Right') & df['HandRT_ms'].notna()]
df_left = df[(df['Direction'] == 'Left') & df['HandRT_ms'].notna()]

# Common bins across both directions
all_rts = df['HandRT_ms'].dropna().values
bins = np.histogram_bin_edges(all_rts, bins=20)

# Plot 1: Right vs Left (Left as negative)
plt.figure(figsize=(8, 4))
plt.hist(df_right['HandRT_ms'], bins=bins, color='C0', alpha=0.7, edgecolor='black', label='Right')
counts_left, _ = np.histogram(df_left['HandRT_ms'], bins=bins)
bin_centers = 0.5 * (bins[1:] + bins[:-1])
bin_widths = bins[1:] - bins[:-1]
plt.bar(bin_centers, -counts_left, width=bin_widths, color='C1', alpha=0.7, edgecolor='black', label='Left')
plt.title('HandRT_ms Histogram for BlockType = S (Right vs Left)')
plt.xlabel('HandRT_ms')
plt.ylabel('Count (Left shown negative)')
plt.axhline(0, color='black', linewidth=1)
plt.legend()
plt.tight_layout()
plt.show()

# Plot 2: All RTs combined (same style, positive counts)
plt.figure(figsize=(8, 4))
plt.hist(all_rts, bins=bins, color='C0', alpha=0.7, edgecolor='black')
plt.title('HandRT_ms Histogram for BlockType = S (All Directions)')
plt.xlabel('HandRT_ms')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


### PyDDM: 3-parameter DDM for a 2-choice task
We pool the two directions as the RTs are similar. We fit a simplest 3-parameter DDM: drift `v`, bound `a`, and a non-decision time `t0`. The noise is fixed at 1.


In [ ]:
import pandas as pd
import numpy as np

# Ensure PyDDM is available
import pyddm
from pyddm import Model
from pyddm.models import DriftConstant, NoiseConstant, BoundConstant, OverlayNonDecision, ICPointSourceCenter, LossRobustLikelihood
from pyddm import Fittable, Sample

# use the same min and max parameter values across fits
min_drift_rate = -10
max_drift_rate = 20
min_bound = 0.2
max_bound = 3
min_ndt = 0.1
max_ndt = 1

df = pd.read_csv(r"C:\Users\Rishv\Desktop\SNL Lab\Participant Data\CMT0015_MASTER_Summary.csv")
df = df[df['BlockType'] == 'S'].copy()

# Pool Left and Right responses: treat all trials as 'correct'
# Convert RTs to seconds for PyDDM
df = df[df['HandRT_ms'].notna()]
df['rt_s'] = df['HandRT_ms'] / 1000.0
df['choice'] = 1  # all responses treated as correct

# Build PyDDM Sample (single-choice data)
sample = Sample.from_pandas_dataframe(df, rt_column_name='rt_s', choice_column_name='choice')

# Simplest 3-parameter DDM (v, a, t0), fix noise s=1
model = Model(
    drift=DriftConstant(drift=Fittable(minval=min_drift_rate, maxval=max_drift_rate)),
    noise=NoiseConstant(noise=1),
    bound=BoundConstant(B=Fittable(minval=min_bound, maxval=max_bound)),
    overlay=OverlayNonDecision(nondectime=Fittable(minval=min_ndt, maxval=max_ndt)),
    IC=ICPointSourceCenter(),
    dx=0.01,
    dt=0.001,
    T_dur=5.0
)

# Fit the model
fit_model = model.fit(sample, lossfunction=LossRobustLikelihood)
fit_model


In [ ]:
# Plot predicted RT distribution over histogram
# Assumes 'fit_model' and 'sample' from prior cell

# Solve the fitted model
sol = model.solve()
t = model.t_domain()

# Predicted PDFs for upper (choice=1) and lower (choice=0) bounds
pdf_upper = sol.pdf('correct')
pdf_lower = sol.pdf('error')

# Mixture weight based on empirical choice proportions
p_upper = (df['choice'] == 1).mean()
p_lower = 1 - p_upper

pdf_mix = p_upper * pdf_upper + p_lower * pdf_lower

# Empirical RTs in seconds
rts = df['rt_s'].values

params = model.parameters()
thedrift = params.get('drift', None)
thebound = params.get('bound', None)
nondectime = params.get('overlay', None)

print(f"Fitted drift rate: {thedrift}\nbound: {thebound}\nndt: {nondectime}")

plt.figure(figsize=(8, 4))
plt.hist(rts, bins=20, density=True, alpha=0.5, edgecolor='black', label='Empirical')
plt.plot(t, pdf_mix, color='red', linewidth=2, label='DDM predicted')
plt.title('Empirical RT Histogram vs DDM Predicted PDF')
plt.xlabel('RT (s)')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.xlim(0.180, 0.5)
plt.show()


### PyDDM fits for BlockType = I (Speed 0, 75, 150)
We pool Left/Right responses and fit separate models for each Speed.


In [ ]:
import pandas as pd
import numpy as np

# Ensure PyDDM is available
try:
    import pyddm
    from pyddm import Model
    from pyddm.models import DriftConstant, NoiseConstant, BoundConstant, OverlayNonDecision, ICPointSourceCenter, LossRobustLikelihood
    from pyddm import Fittable, Sample
except Exception as e:
    raise ImportError("PyDDM is not installed or failed to import. Install with: pip install pyddm") from e

def fit_pooled_ddm_for_speed(df_i, speed):
    df_speed = df_i[(df_i['Speed_deg_per_s'] == speed) & df_i['HandRT_ms'].notna()].copy()
    df_speed['rt_s'] = df_speed['HandRT_ms'] / 1000.0
    df_speed['choice'] = 1  # pooled, treat all as correct

    sample = Sample.from_pandas_dataframe(df_speed, rt_column_name='rt_s', choice_column_name='choice')

    model = Model(
        drift=DriftConstant(drift=Fittable(minval=min_drift_rate, maxval=max_drift_rate)),
        noise=NoiseConstant(noise=1),
        bound=BoundConstant(B=Fittable(minval=min_bound, maxval=max_bound)),
        overlay=OverlayNonDecision(nondectime=Fittable(minval=min_ndt, maxval=max_ndt)),
        IC=ICPointSourceCenter(),
        dx=0.01,
        dt=0.01,
        T_dur=5.0
    )

    fit_model = model.fit(sample, lossfunction=LossRobustLikelihood)
    return df_speed, sample, fit_model, model

df = pd.read_csv(r"C:\Users\Rishv\Desktop\SNL Lab\Participant Data\CMT0015_MASTER_Summary.csv")
df_i = df[df['BlockType'] == 'I'].copy()

df_i_0, sample_i_0, fit_model_i_0, model_i_0 = fit_pooled_ddm_for_speed(df_i, 0)
df_i_75, sample_i_75, fit_model_i_75, model_i_75 = fit_pooled_ddm_for_speed(df_i, 75)
df_i_150, sample_i_150, fit_model_i_150, model_i_150 = fit_pooled_ddm_for_speed(df_i, 150)

fit_model_i_0, fit_model_i_75, fit_model_i_150


In [ ]:
# Plot empirical RT histograms vs DDM predicted PDFs for each Speed (BlockType = I)
import numpy as np
import matplotlib.pyplot as plt

def plot_empirical_vs_ddm(df_speed, model, title, bins):
    sol = model.solve()
    t = model.t_domain()

    pdf_pred = sol.pdf('correct')

    all_rts = df_speed['HandRT_ms'].dropna().values

    plt.figure(figsize=(8, 4))
    plt.hist(all_rts / 1000.0, bins=bins / 1000.0, density=True, alpha=0.5, edgecolor='black', label='Empirical')
    plt.plot(t, pdf_pred, color='red', linewidth=2, label='DDM predicted')
    plt.title(title)
    plt.xlabel('RT (s)')
    plt.ylabel('Density')
    plt.legend()
    plt.tight_layout()
    plt.xlim(0.180, 0.5)
    plt.show()

# Common bins across all three speeds
all_rts_combined = np.concatenate([
    df_i_0['HandRT_ms'].dropna().values,
    df_i_75['HandRT_ms'].dropna().values,
    df_i_150['HandRT_ms'].dropna().values,
])
common_bins = np.histogram_bin_edges(all_rts_combined, bins=20)

plot_empirical_vs_ddm(df_i_0, model_i_0, 'BlockType=I, Speed=0', common_bins)
plot_empirical_vs_ddm(df_i_75, model_i_75, 'BlockType=I, Speed=75', common_bins)
plot_empirical_vs_ddm(df_i_150, model_i_150, 'BlockType=I, Speed=150', common_bins)


In [ ]:
# Summary table of fitted DDM parameters for 4 cases
import pandas as pd
import numpy as np

def _val(x):
    if hasattr(x, 'value'):
        return x.value
    if hasattr(x, 'val'):
        return x.val
    return x

def _params_dict(model):
    if model is None:
        return {}
    params = model.get_model_parameters()
    if isinstance(params, dict):
        return params
    names = model.get_model_parameter_names()
    return dict(zip(names, params))

def extract_params(model, label):
    params = _params_dict(model)
    return {
        'Case': label,
        'drift': _val(params.get('drift', np.nan)),
        'bound': _val(params.get('B', np.nan)),
        'nondectime': _val(params.get('nondectime', np.nan))
    }

rows = [
    extract_params(globals().get('model', None), 'BlockType=S (pooled)'),
    extract_params(globals().get('model_i_0', None), 'BlockType=I, Speed=0'),
    extract_params(globals().get('model_i_75', None), 'BlockType=I, Speed=75'),
    extract_params(globals().get('model_i_150', None), 'BlockType=I, Speed=150')
]

param_table = pd.DataFrame(rows)
param_table
